In [1]:
import os

In [2]:
file = 'data/day15.txt'
path = os.path.join(os.getcwd(), file)
fp = open(path, 'r')
rows = [x.strip('\n') for x in fp.readlines()]

In [3]:
data = []
for row in rows: # sens. row, sens. col, beacon row, beacon col, manhattan dist.
    row = row.replace('Sensor at x=', '').replace(' y=', '')
    row = row.replace(': closest beacon is at x=', ',')
    row = [int(coord) for coord in row.split(',')]
    row.append(abs(row[0] - row[2]) + abs(row[1] - row[3]))
    data.append(row)

In [4]:
# part one
test_row, exclude_set = 2000000, set()
beacons = {}
for elem in data:
    beacons[elem[3]] = beacons.get(elem[3], []) + [elem[2]]
for elem in data:
    test_row_dist = abs(test_row - elem[1])
    test_row_offset, manhattan_distance = elem[0], elem[4]
    test_row_width = (2 * manhattan_distance + 1) - (2 * test_row_dist)
    if test_row_width > 0:
        lower = test_row_offset - (test_row_width // 2)
        upper = test_row_offset + (test_row_width // 2)
        exclude_set.update(list(range(lower, upper + 1)))
    if test_row in beacons:
        exclude_set = exclude_set.difference(set(beacons[test_row]))
print(len(exclude_set))

5100463


In [5]:
# part two
down_right_lines, up_right_lines = [], []
for elem in data: # creates lists of sensor perimeter lines
    sensor_coord = (elem[0], elem[1])
    m_dist = elem[4]
    north = (sensor_coord[0], sensor_coord[1] + m_dist + 1)
    south = (sensor_coord[0], sensor_coord[1] - m_dist - 1)
    east = (sensor_coord[0] + m_dist + 1, sensor_coord[1])
    west = (sensor_coord[0] - m_dist - 1, sensor_coord[1])
    down_right_lines.extend([(north, east), (south, west)])
    up_right_lines.extend([(east, south), (west, north)])

In [6]:
intersections = set()
x_min, y_min = 0, 0
x_max, y_max = 4000000, 4000000
for d in down_right_lines: # finds the intersections of all perimeter lines
    for u in up_right_lines:
        down_y_inter, up_y_inter = d[0][1] + d[0][0], u[0][1] - u[0][0]
        y_inters = down_y_inter - up_y_inter
        if not y_inters % 2:
            x = y_inters // 2
            inter = (x, x + up_y_inter)
            x_low = max([d[0][0], u[0][0], x_min])
            x_high = min([d[1][0], u[1][0], x_max])
            y_low = max([d[1][1], u[0][1], y_min])
            y_high = min([d[0][1], u[1][1], y_max])
            if x_low <= inter[0] <= x_high and y_low <= inter[1] <= y_high:
                intersections.add(inter)

In [7]:
for inter in intersections.copy():
    for elem in data: # finds the valid distress signal beacon position
        s_coord = (elem[0], elem[1])
        s_m_dist = elem[4]
        s_inter_m_dist = abs(s_coord[0] - inter[0]) + abs(s_coord[1] - inter[1])
        if s_inter_m_dist <= s_m_dist:
            intersections.remove(inter)
            break
d_beacon_coordinate = list(intersections).pop()
tuning_freq = d_beacon_coordinate[0] * 4000000 + d_beacon_coordinate[1]
print(tuning_freq)

11557863040754
